In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import Dataset, DataLoader

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
df = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, train_size = 0.8, random_state=42)
train_df = train_df.to_numpy()
test_df = test_df.to_numpy()
y_train, x_train = train_df[:,0], train_df[:,1:].astype(np.float32)
y_test, x_test = test_df[:,0], test_df[:,1:].astype(np.float32)

In [ ]:
x_train, y_train, x_test, y_test = map(torch.tensor, (x_train, y_train, x_test, y_test))

In [ ]:
x_train /= 255.0
x_test  /= 255.0

In [ ]:
eps = 1e-6
def relu(x):
    return x.clamp(min=0)

def batch_norm(x, gamma):
    batch_mean = x.mean(dim=0)
    batch_var = (x-batch_mean).var(dim=0)
    x_norm = (x-batch_mean) / torch.sqrt(batch_var + eps)
    return gamma * x_norm

In [ ]:
BATCH_SIZE = 64
epochs = 300
EPOCHS_PER_EVAL = 10
lr = 1e-2
hidden_features = 512

In [ ]:
class MNIST(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.n = len(x)

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

train_ds = MNIST(x_train, y_train)
test_ds = MNIST(x_test, y_test)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE)

In [ ]:
x_train, y_train, x_test, y_test = [t.to(device) for t in [x_train, y_train, x_test, y_test]]
w1 = torch.distributions.Uniform(-np.sqrt(1/784), np.sqrt(1/784)).sample((hidden_features,784)).to(device).requires_grad_()
b1 = torch.distributions.Uniform(-np.sqrt(1/784), np.sqrt(1/784)).sample((hidden_features,)).to(device).requires_grad_()
w2 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((hidden_features,hidden_features)).to(device).requires_grad_()
b2 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((hidden_features,)).to(device).requires_grad_()
w3 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((10,hidden_features)).to(device).requires_grad_()
b3 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((10,)).to(device).requires_grad_()

In [ ]:
def cross_entropy_loss(y_vals, y_true):
    log_probs = torch.log_softmax(y_vals, dim=1)
    return -log_probs[range(y_true.shape[0]), y_true].mean()

In [ ]:
def forward(x):
    a1 = relu(x @ w1.T + b1)
    a2 = relu(a1 @ w2.T + b2)
    y_vals = a2 @ w3.T + b3
    return y_vals

def update_weights(lr):
    with torch.no_grad():
        for param in [w1, b1, w2, b2, w3, b3]:
            param -= lr * param.grad
            param.grad.zero_()

In [ ]:
def evaluate():
    total = 0
    correct = 0
    total_loss = 0
    for x_batch, y_batch in test_dl:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        y_vals = forward(x_batch)
        loss = cross_entropy_loss(y_vals, y_batch)
        total_loss += loss.item()
        correct += (y_vals.argmax(1) == y_batch).sum().item()
        total += y_batch.size(0)
    return total_loss / len(test_dl), correct / total * 100

In [ ]:
for epoch in range(epochs):
    total_loss = 0
    correct = 0
    total = 0
    for x_batch, y_batch in train_dl:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        y_vals = forward(x_batch)
        loss = cross_entropy_loss(y_vals, y_batch)
        loss.backward()
        update_weights(lr)
        total_loss += loss.item()
        correct += (y_vals.argmax(1) == y_batch).sum().item()
        total += y_batch.size(0)
    
    val_loss, val_acc = evaluate()

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {total_loss/len(train_dl):.4f}, "
          f"Train Acc: {correct/total*100:.2f}% | "
          f"Val Loss: {val_loss:.4f}, "
          f"Val Acc: {val_acc:.2f}%")